# Lightning AI Training Notebook: Hana Kim (EA-F-001) Flux.1-dev LoRA
This notebook automates setting up the workspace on a Lightning AI Studio and running the LoRA training using `ostris/ai-toolkit` for Flux.1-dev.

### Step 1: Install Dependencies
Run this cell in your Lightning AI Studio to clone the trainer repository and install python requirements.

In [ ]:
# Clone the trainer repo into your Teamspace Studio
!git clone --recursive https://github.com/ostris/ai-toolkit.git /teamspace/studios/this_studio/ai-toolkit
%cd /teamspace/studios/this_studio/ai-toolkit

# Install dependencies
!pip install -r requirements.txt
!pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu121

### Step 1.5: Patch System torchaudio Conflicts
Run this cell to patch the code and prevent the broken system `torchaudio` package from failing your run. Since `torchaudio` is not used in Flux LoRA image training, we can safely bypass it.

In [ ]:
import os

toolkit_dir = "/teamspace/studios/this_studio/ai-toolkit"
patched = 0
for root, dirs, files in os.walk(toolkit_dir):
    for file in files:
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()
                if "import torchaudio" in content:
                    print(f"Patching: {filepath}")
                    content = content.replace("import torchaudio", "try:\n    import torchaudio\nexcept Exception:\n    torchaudio = None")
                    with open(filepath, "w", encoding="utf-8") as f:
                        f.write(content)
                    patched += 1
            except Exception as e:
                pass

print(f"Completed patching. Total files patched: {patched}")

### Step 2: Upload Dataset
Upload the `character_id` directory to `/teamspace/studios/this_studio/character_id` using the Lightning AI sidebar file manager. Alternatively, run this cell to initialize the directories.

In [ ]:
import os
os.makedirs("/teamspace/studios/this_studio/character_id/train", exist_ok=True)
os.makedirs("/teamspace/studios/this_studio/character_id/identity_anchors", exist_ok=True)
print("Created directories on Lightning AI. Please drag and drop your images/captions into /teamspace/studios/this_studio/character_id/train/")

### Step 3: Create Training Configuration
Run this cell to write the `flux_training_config.yaml` file pointing to your dataset in the Lightning AI Teamspace.

In [ ]:
yaml_config = """
job: extension
config:
  name: "ea_f_001_hana_kim_lora"
  process:
    - type: 'sd_trainer'
      training_folder: "/teamspace/studios/this_studio/output"
      performance_log_every: 250
      device: cuda
      trigger_word: "ea_f_001_hana_kim"
      network:
        type: "lora"
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 500
        max_step_saves_to_keep: 4
      datasets:
        - folder_path: "/teamspace/studios/this_studio/character_id/train"
          caption_ext: "txt"
          resolution: [512, 768, 1024]
      train:
        epochs: 45
        steps: 2500
        gradient_accumulation_steps: 1
        learning_rate: 1e-4
        batch_size: 1
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw8bit"
        ema_decay: 0.99
        dtype: bf16
      model:
        name_or_path: "black-forest-labs/FLUX.1-dev"
        is_flux: true
        quantize: true
      sample:
        sampler: "flowmatch"
        sample_every: 250
        width: 1024
        height: 1024
        prompts:
          - "ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, wearing elegant red halterneck dress, outdoor garden background, high fashion editorial, 85mm portrait"
          - "ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, wearing cream sleeveless knit dress, luxury studio background, direct gaze, soft lighting"
"""

config_dir = "/teamspace/studios/this_studio/ai-toolkit/config"
os.makedirs(config_dir, exist_ok=True)
with open(os.path.join(config_dir, "flux_training_config.yaml"), "w") as f:
    f.write(yaml_config)
print("Training configuration written to /teamspace/studios/this_studio/ai-toolkit/config/flux_training_config.yaml")

### Step 4: Run Training
Ensure your Studio is using a GPU environment (like an L4, A10G, or A100 GPU) in the Lightning AI interface, then run this cell to start the training process.

In [ ]:
%cd /teamspace/studios/this_studio/ai-toolkit
!python run.py config/flux_training_config.yaml

### Step 5: Export Model
Once training is complete, your LoRA weight file (`ea_f_001_hana_kim_lora.safetensors`) will be generated inside `/teamspace/studios/this_studio/output/ea_f_001_hana_kim_lora`. You can download it directly to your machine or store it in your teamspace files.